In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
from src import utils

In [3]:
from huggingface_hub import HfFolder, login

api_file = "/home/fre.gilad/source/llm-iml/HF_KEY.txt"
hf_token = utils.api_key_from_file(api_file)

HfFolder.save_token(hf_token)
login(token=hf_token)

In [4]:
import pandas as pd
from src.data import DF_Batcher

data = pd.read_csv("/home/fre.gilad/source/llm-iml/data/HarmBench/harmful_behaviors.csv")
data = data.rename(columns={"goal": "prompt"})

ds_train = data.copy()
ds_eval = data.copy()

dl_train = DF_Batcher(ds_train, batch_size=10, shuffle=True)
dl_eval = DF_Batcher(ds_eval, batch_size=50, shuffle=False)

In [5]:
print("Train dataset size:", len(ds_train))
print("Eval dataset size:", len(ds_eval))

Train dataset size: 200
Eval dataset size: 200


In [6]:
from src.eval.harmbench_evaluator import HarmbenchEvaluator
from src.eval.llama_evaluator import LlamaEvaluator
from src.eval.template_evaluator import TemplateEvaluator
from src.eval.llama_guard_evaluator import LlamaGuardEvaluator
from src.eval.strong_reject_evaluator import StrongRejectEvaluator
from gserve.configs import ServeConfig, LLMConfig


evaluators = [
    HarmbenchEvaluator(
        serve_config=ServeConfig(gpu_ids=[1], startup_timeout=10 * 60, client_timeout=60, verbose=True),
        use_context=False,
        silent=False,
    ),
    # LlamaGuardEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     model_name="meta-llama/Meta-Llama-Guard-2-8B",
    #     silent=False,
    # ),
    # StrongRejectEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     # binary_thresh=0.5,
    #     silent=False,
    # ),
    # LlamaEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     silent=False,
    # ),
    TemplateEvaluator(
        silent=False,
    ),
]

INFO 06-12 13:21:40 [__init__.py:243] Automatically detected platform cuda.


INFO 06-12 13:21:44 [vllm_service.py:153] Launching subprocess:
    /home/fre.gilad/source/llm-iml/.venv/bin/python /home/fre.gilad/source/llm-iml/.venv/lib/python3.12/site-packages/gserve/vllm_server.py --serve --model cais/HarmBench-Llama-2-13b-cls --host 127.0.0.1 --port 42541 --gpus 1 --llm_kwargs {"dtype": "bfloat16", "tokenizer_mode": "auto", "trust_remote_code": false, "seed": 0, "enforce_eager": false}


INFO 06-12 13:21:50 [__init__.py:243] Automatically detected platform cuda.


INFO 06-12 13:21:53 [vllm_server.py:196] Initializing LLM 'cais/HarmBench-Llama-2-13b-cls' on GPUs 1


INFO 06-12 13:21:53 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 06-12 13:21:53 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 06-12 13:21:53 [__init__.py:36] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 06-12 13:22:05 [config.py:793] This model supports multiple tasks: {'embed', 'reward', 'classify', 'generate', 'score'}. Defaulting to 'generate'.
INFO 06-12 13:22:05 [config.py:2118] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-12 13:22:08 [core.py:438] Waiting for init message from front-end.
INFO 06-12 13:22:08 [core.py:65] Initializing a V1 LLM engine (v0.9.0.1) with config: model='cais/HarmBench-Llama-2-13b-cls', speculative_config=None, tokenizer='cais/HarmBench-Llama-2-13b-cls', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, tr

Loading safetensors checkpoint shards:   0% Completed | 0/6 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  17% Completed | 1/6 [00:00<00:01,  4.66it/s]
Loading safetensors checkpoint shards:  33% Completed | 2/6 [00:01<00:02,  1.53it/s]
Loading safetensors checkpoint shards:  50% Completed | 3/6 [00:02<00:02,  1.11it/s]
Loading safetensors checkpoint shards:  67% Completed | 4/6 [00:03<00:02,  1.04s/it]
Loading safetensors checkpoint shards:  83% Completed | 5/6 [00:04<00:01,  1.08s/it]
Loading safetensors checkpoint shards: 100% Completed | 6/6 [00:05<00:00,  1.10s/it]
Loading safetensors checkpoint shards: 100% Completed | 6/6 [00:05<00:00,  1.02it/s]



INFO 06-12 13:22:17 [default_loader.py:280] Loading weights took 6.08 seconds
INFO 06-12 13:22:17 [gpu_model_runner.py:1549] Model loading took 24.2836 GiB and 7.522940 seconds
INFO 06-12 13:22:27 [backends.py:459] Using cache directory: /home/fre.gilad/.cache/vllm/torch_compile_cache/9ae5f18dfb/rank_0_0 for vLLM's torch.compile
INFO 06-12 13:22:27 [backends.py:469] Dynamo bytecode transform time: 9.83 s
INFO 06-12 13:22:34 [backends.py:132] Directly load the compiled graph(s) for shape None from the cache, took 6.273 s
INFO 06-12 13:22:35 [monitor.py:33] torch.compile takes 9.83 s in total
INFO 06-12 13:22:42 [kv_cache_utils.py:637] GPU KV cache size: 22,208 tokens
INFO 06-12 13:22:42 [kv_cache_utils.py:640] Maximum concurrency for 2,048 tokens per request: 10.84x
INFO 06-12 13:23:17 [gpu_model_runner.py:1933] Graph capturing finished in 35 secs, took 1.78 GiB
INFO 06-12 13:23:17 [core.py:167] init engine (profile, create kv cache, warmup model) took 60.13 seconds


INFO 06-12 13:23:18 [vllm_server.py:203] LLM initialized, starting server at 127.0.0.1:42541
INFO:     Started server process [1971426]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:42541 (Press CTRL+C to quit)
INFO 06-12 13:23:19 [vllm_service.py:210] Server is healthy at http://127.0.0.1:42541/health
INFO 06-12 13:23:19 [vllm_service.py:402] Started 1 server(s) listening on 127.0.0.1:42541


INFO:     127.0.0.1:58326 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:58334 - "GET /health HTTP/1.1" 200 OK


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

torch.set_float32_matmul_precision("high")  # negligable effect

# model_name = "Qwen/Qwen3-0.6B"
# model_name = "GraySwanAI/Llama-3-8B-Instruct-RR"
# model_name = "meta-llama/Llama-3.2-1B-Instruct"
# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
model_name = "meta-llama/Llama-2-7b-chat-hf"
# model_name = "lmsys/vicuna-7b-v1.5" # TODO: not instruct model, no chat template
# model_name = "mistralai/Mistral-7B-Instruct-v0.3"
# model_name = "tiiuae/falcon-7b-instruct"
# model_name = "mosaicml/mpt-7b-chat"
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

utils.set_seed(42)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="sequential",
    # attn_implementation="flash_attention_2"
    # attn_implementation="sdpa",
)

if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
    (rotary_e

In [9]:
from torch import optim
from src.activation_extractor import ActivationExtractor
from src.attacks.optim_attack import OptimAttack
from src.iml_attack import IML_Attack, StopCriteria
from src.adver_model import AdverModel
from src.initialize import Initializer


adv_model = AdverModel(
    model=model,
    tokenizer=tokenizer,
    num_tokens=20
)

Initializer.normal(adv_model)

internal_attack = OptimAttack(
    adv_model,
    optim_factory=lambda params: optim.AdamW(params, lr=1e-3),
    steps=30,
    mixed_precision=False,
)

iml_attack = IML_Attack(
    adv_model=adv_model,
    internal_attack=internal_attack,
    optim_factory=lambda params: optim.AdamW(params, lr=2e-2),
    evaluators=evaluators,
    pred_kwargs={"max_length": 512},
    mixed_precision=False,
)

stop = StopCriteria(
    max_epochs=5,
    max_time=15 * 60,
    patience=3,
)

In [ ]:
adv_model = iml_attack.fit(dl_train, dl_eval, stop_criteria=stop)

Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

Predict:   0%|          | 0/4 [00:00<?, ?it/s]

Evaluating cais/HarmBench-Llama-2-13b-cls:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:     127.0.0.1:42678 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:42678 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:42678 - "POST /generate HTTP/1.1" 200 OK
INFO:     127.0.0.1:42678 - "POST /generate HTTP/1.1" 200 OK


Batch:   0%|          | 0/20 [00:00<?, ?it/s]

...

Attack:   0%|          | 0/30 [00:00<?, ?it/s]

Attack:   0%|          | 0/30 [00:00<?, ?it/s]

Attack:   0%|          | 0/30 [00:00<?, ?it/s]

Attack:   0%|          | 0/30 [00:00<?, ?it/s]

Attack:   0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
# TODO: make sure the model computation actually runs at the model dtype
adv_model.set_embeddings(iml_attack.best_embeds)
iml_attack.evaluate(adv_model=adv_model, dl_eval=dl_eval, evalers=evaluators)

In [ ]:
adv_model.set_embeddings(iml_attack.best_embeds)
preds = iml_attack.predict(adv_model, dl_eval, max_length=300)
dl_eval.set_column("response", preds)

for i in range(len(preds)):
    print(f" == Prompt:")
    print(ds_eval.iloc[i]["prompt"])
    print(f" == Target:")
    print(ds_eval.iloc[i]["target"])
    print(f" == Prediction:")
    print(preds[i])
    print("\n" + "=" * 50 + "\n")